In [1]:
%pip install numpy pandas matplotlib scikit-learn jinja2 seaborn

Note: you may need to restart the kernel to use updated packages.


error: externally-managed-environment

× This environment is externally managed
╰─> This Python installation is managed by uv and should not be modified.

note: If you believe this is a mistake, please contact your Python installation or OS distribution provider. You can override this, at the risk of breaking your Python installation or OS, by passing --break-system-packages.
hint: See PEP 668 for the detailed specification.


In [3]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.preprocessing import PolynomialFeatures, StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score
from sklearn.neighbors import KNeighborsClassifier
from sklearn.preprocessing import StandardScaler


RANDOM_STATE = 42

ModuleNotFoundError: No module named 'numpy'

In [4]:
df = pd.read_csv('.../DATA_Problematic Mobile Phone Use CAT.csv')
df.head()

NameError: name 'pd' is not defined

In [ ]:
print('Shape:', df.shape)
print('\nData types:')
print(df.dtypes)

print('\nMissing values:')
print(df.isna().sum())

df.describe().round(2)

In [ ]:
df['nmp_info'] = df[['NMP_1', 'NMP_2', 'NMP_3', 'NMP_4']].sum(axis=1)
df['nmp_conv'] = df[['NMP_5', 'NMP_6', 'NMP_7', 'NMP_8']].sum(axis=1)
df['nmp_cont'] = df[['NMP_9', 'NMP_10', 'NMP_11', 'NMP_12']].sum(axis=1)
df['nmp_conn'] = df[['NMP_13', 'NMP_14', 'NMP_15', 'NMP_16']].sum(axis=1)
df['nmp_total'] = df[['nmp_info', 'nmp_conv', 'nmp_cont', 'nmp_conn']].sum(axis=1)

df['nmp_norm'] = (
    (df['nmp_total'] - df['nmp_total'].min()) /
    (df['nmp_total'].max() - df['nmp_total'].min())
)

# SPAI subscales
df['spai_comp'] = df[['SPAI_3', 'SPAI_4', 'SPAI_5', 'SPAI_8', 'SPAI_9', 'SPAI_10', 'SPAI_15', 'SPAI_17']].sum(axis=1)
df['spai_imp'] = df[['SPAI_6', 'SPAI_10', 'SPAI_11', 'SPAI_12', 'SPAI_14', 'SPAI_18', 'SPAI_19']].sum(axis=1)
df['spai_with'] = df[['SPAI_2', 'SPAI_13', 'SPAI_16', 'SPAI_20']].sum(axis=1)
df['spai_tol'] = df[['SPAI_1', 'SPAI_7']].sum(axis=1)

# SPAI total
df['spai_total'] = df[['spai_comp', 'spai_imp', 'spai_with', 'spai_tol']].sum(axis=1)

# Normalise SPAI total (0-1)
df['spai_norm'] = (
    (df['spai_total'] - df['spai_total'].min()) /
    (df['spai_total'].max() - df['spai_total'].min())
)

print('X shape:', X.shape)
print('y shape:', y.shape)

In [ ]:
#df['nmp-total'].hist(bins=20)
#%pip install seaborn
print('Mean:', df['nmp_norm'].mean())
sns.histplot(data=df, x='nmp_norm', bins=11, kde=True)

plt.xlabel('NMP Standardised Score')
plt.ylabel('Frequency')
plt.title('Distribution of NMP Standardised Score')
plt.xticks([0,0.5,1],
           ['Abscence of NMP', 'Neutral', 'Severe NMP']
           )
plt.axvline(df['nmp_norm'].mean(), color='red', linestyle='--')
plt.show()

In [ ]:
grade_mapping = {
    11: 1, 12: 2, 13: 3, 14: 4, 15: 5, 16: 6, #Grades 7-12
    1: 7, 2: 8, 3: 9, 4: 10, #Fresh-Senior
    5: 11, 6: 12, 7: 13, #Masters 1,2,3
    8: 14 #Doctor
}



df['grade_ord'] = df['grade'].map(grade_mapping)
df = df[df['grade_ord'] > 3]


df['grade_ord'].hist(bins=11)

df['grade_ord'].value_counts().sort_index()

df['grade_binary'] = np.where(df['grade'] > 10, 1, 0)
df['grade_binary'].value_counts()

In [ ]:
grade_mapping2 = {
    4:1, 5:1, 6:1, #Grades 9-12
    7:2, 8:2, 9:2, 10:2, #Fresh-Senior
    11:3, 12:3, 13:3, #Masters 1,2,3
    14:4 #Doctor
}

df['grade_ord_binned'] = df['grade_ord'].map(grade_mapping2)

df['grade_ord_binned'].hist(bins=4)

df['grade_ord_binned'].value_counts().sort_index()

In [ ]:

sns.histplot(data=df, x='nmp_norm', bins=20, kde=True)

plt.xlabel('NMP')
plt.ylabel('Frequency')
plt.title('Distribution of NMP')

plt.show()

In [ ]:

plt.scatter(df['nmp_norm'], df['grade_ord_binned'],
             alpha=0.3
            )

# Label the y-axis
plt.ylabel('grade (binned)')

# Set 4 y-axis ticks and their labels
plt.yticks(
    [1, 2, 3, 4],
    ['Secondary', 'College', 'Masters', 'Doctorate']
)

# Label the x-axis
plt.xlabel('NMP score')

# Set 3 x-axis ticks and their labels
plt.xticks(
    [0,0.5,1],
    ['Absence of nomophobia', 'Neutral', 'Severe Nomophobia']
)

In [ ]:
X = df[["nmp_norm", "spai_norm"]]
y = df["grade_binary"]

class_names = {
    0: "Fail",
    1: "Pass"
}

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.25,
    random_state=RANDOM_STATE,
    stratify=y
)

print("Training observations:", len(X_train))
print("Test observations:", len(X_test))

# Share of Pass/Fail in each group; stratify=y keeps these similar
pd.DataFrame({
    "All students": y.value_counts(normalize=True),
    "Training set": y_train.value_counts(normalize=True),
    "Test set": y_test.value_counts(normalize=True)
}).rename(index=class_names).round(2)

In [ ]:
log_reg = make_pipeline(
    StandardScaler(),
    LogisticRegression(random_state=RANDOM_STATE)
)

log_reg.fit(X_train, y_train)

log_predictions = log_reg.predict(X_test)
log_probabilities = log_reg.predict_proba(X_test)[:, 1]

print("First 10 predicted classes:", log_predictions[:10])
print("First 10 probabilities of Pass:", np.round(log_probabilities[:10], 2))
print("Test accuracy:", round(accuracy_score(y_test, log_predictions), 3))

In [ ]:
knn = make_pipeline(
    StandardScaler(),
    KNeighborsClassifier(n_neighbors=5)
)

knn.fit(X_train, y_train)
knn_predictions = knn.predict(X_test)

print("First 10 predicted classes:", knn_predictions[:10])
print("Test accuracy:", round(accuracy_score(y_test, knn_predictions), 3))

In [ ]:
results = pd.DataFrame({
    "Model": ["Logistic Regression", "KNN (k=5)"],
    "Test accuracy": [
        accuracy_score(y_test, log_predictions),
        accuracy_score(y_test, knn_predictions)
    ]
}).sort_values("Test accuracy", ascending=False)

results

In [ ]:
def plot_decision_boundary(model, X_train, y_train, title, ax):

    x_min, x_max = X_train["nmp_norm"].min() - 0.5, X_train["nmp_norm"].max() + 0.5

    y_min, y_max = X_train["spai_norm"].min() - 0.5, X_train["spai_norm"].max() + 0.5

    xx, yy = np.meshgrid(
        np.linspace(x_min, x_max, 250),
        np.linspace(y_min, y_max, 250)
    )

    grid = pd.DataFrame({
        "nmp_norm": xx.ravel(),
        "spai_norm": yy.ravel()
    })

    zz = model.predict(grid).reshape(xx.shape)

    ax.contourf(xx, yy, zz, alpha=0.18, levels=[-0.5, 0.5, 1.5])

    for label, name in [(0, "Fail"), (1, "Pass")]:
        subset = X_train.loc[y_train == label]
        ax.scatter(
            subset["nmp_norm"],
            subset["spai_norm"],
            label=name,
            alpha=0.8
        )

    ax.set_xlabel("Normalized Study Hours")
    ax.set_ylabel("Normalized Contribution")
    ax.set_title(title)


fig, axes = plt.subplots(1, 2, figsize=(13, 5), sharex=True, sharey=True)

plot_decision_boundary(log_reg, X_train, y_train, "Logistic Regression", axes[0])
plot_decision_boundary(knn, X_train, y_train, "KNN (k=5)", axes[1])

axes[0].legend()
plt.tight_layout()
plt.show()